# Assignment 2: Neural Sequence Models

This notebook implements the **Bidirectional LSTM** part of Assignment 2 for the caption–question relevance task on RSVLM-QA, and provides scaffolding for adding a Transformer/BERT model later.

We reuse the same dataset, splits, and primary metric (F1-macro) as in Assignment 1.

## 1. Setup and Imports

Set random seeds, import libraries, and configure the device (CPU/GPU).

In [13]:
import math
import random
import time
from collections import Counter
from typing import Dict, List, Tuple

import numpy as np
import pandas as pd
import torch
import torch.nn as nn
from sklearn.metrics import f1_score
from sklearn.model_selection import train_test_split
from torch.utils.data import DataLoader, Dataset

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Using device:", device)

def set_seed(seed: int = 42):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False

set_seed(42)

Using device: cpu



## 2. Load RSVLM-QA data and construct the classification dataset

We replicate the dataset from Assignment 1:
- **Positive pairs (label=1)**: correct caption–question–answer triplets from the same image
- **Negative pairs (label=0)**: caption from a different image paired with the question–answer
- **Text field**: `caption [SEP] question answer` (same format as A1)


In [14]:

# ---------------------------------------------------------------------------
# Reproduce the EXACT same dataset construction from Assignment 1
# ---------------------------------------------------------------------------
RANDOM_STATE = 42
np.random.seed(RANDOM_STATE)

# 1) Load both Parquet files
df_captions = pd.read_parquet("RSVLM-QA-captions.parquet")
df_qa = pd.read_parquet("RSVLM-QA-questions.parquet")

print("Captions shape:", df_captions.shape, "  columns:", df_captions.columns.tolist())
print("QA shape:      ", df_qa.shape, "  columns:", df_qa.columns.tolist())

# 2) POSITIVE examples: merge captions with QA on shared 'id'
df_positive = df_qa.merge(df_captions[["id", "caption"]], on="id", how="inner")
df_positive["question_answer"] = df_positive["question"].astype(str) + " " + df_positive["answer"].astype(str)
df_positive["label"] = 1

# 3) NEGATIVE examples: pair each QA with a RANDOM distant caption
#    (A1 used tag-embedding semantic distance; random is a safe fallback
#     that still creates valid negatives without needing FastText.)
all_caption_ids = df_captions["id"].values
caption_lookup = df_captions.set_index("id")["caption"]

df_negative = df_qa.copy()
rng = np.random.RandomState(RANDOM_STATE)
neg_captions = []
for _, row in df_qa.iterrows():
    # pick a random caption that does NOT belong to the same image
    while True:
        rand_id = rng.choice(all_caption_ids)
        if rand_id != row["id"]:
            break
    neg_captions.append(caption_lookup[rand_id])

df_negative["caption"] = neg_captions
df_negative["question_answer"] = df_negative["question"].astype(str) + " " + df_negative["answer"].astype(str)
df_negative["label"] = 0

# 4) Combine, shuffle, build the text field matching A1's format
df_combined = pd.concat([df_positive, df_negative], ignore_index=True)
df_combined = df_combined.sample(frac=1, random_state=RANDOM_STATE).reset_index(drop=True)

# Ensure all text columns are strings (handles NaN)
df_combined["caption"] = df_combined["caption"].fillna("").astype(str)
df_combined["question_answer"] = df_combined["question_answer"].fillna("").astype(str)
df_combined["text"] = df_combined["caption"] + " [SEP] " + df_combined["question_answer"]

print(f"\nTotal dataset size: {len(df_combined)}")
print("Label distribution:\n", df_combined["label"].value_counts())
print("\nExample text:", df_combined["text"].iloc[0][:200])


Captions shape: (13820, 4)   columns: ['id', 'image', 'caption', 'tags']
QA shape:       (148558, 4)   columns: ['id', 'question_type', 'question', 'answer']

Total dataset size: 297116
Label distribution:
 label
0    148558
1    148558
Name: count, dtype: int64

Example text: The image primarily features a high-density residential complex with several large apartment buildings arranged in a geometric pattern, occupying much of the central and upper parts of the scene. Road



## 3. Train / Validation / Test split

We first reproduce Assignment 1's **80/20 stratified** train/test split (`random_state=42`), then carve a **10% validation** set from the training portion for early stopping.


In [15]:

# ---------------------------------------------------------------------------
# Replicate A1's 80/20 train/test split, then carve 10% validation from train
# ---------------------------------------------------------------------------
X_all = df_combined["text"].tolist()
y_all = df_combined["label"].tolist()

# Step 1: Exact same 80/20 split as Assignment 1
X_train_full, X_test, y_train_full, y_test = train_test_split(
    X_all, y_all, test_size=0.2, random_state=RANDOM_STATE, stratify=y_all
)

# Step 2: Split training set → 90% train / 10% validation (for early stopping)
X_train, X_val, y_train, y_val = train_test_split(
    X_train_full, y_train_full, test_size=0.1, random_state=RANDOM_STATE, stratify=y_train_full
)

print(f"Train size: {len(X_train)}")
print(f"Val size:   {len(X_val)}")
print(f"Test size:  {len(X_test)}")


Train size: 213922
Val size:   23770
Test size:  59424


## 4. Vocabulary and Tokenizer

We build a simple lowercase whitespace tokenizer and a frequency-based vocabulary with special tokens `<PAD>` and `<UNK>`.

In [16]:
def simple_tokenizer(text) -> List[str]:
    return str(text).lower().split()


def build_vocab(texts: List[str], min_freq: int = 2, max_size: int = 20000) -> Tuple[Dict[str, int], callable]:
    counter: Counter = Counter()
    for t in texts:
        counter.update(simple_tokenizer(t))

    vocab: Dict[str, int] = {"<PAD>": 0, "<UNK>": 1}

    for token, freq in counter.most_common():
        if freq < min_freq:
            continue
        if len(vocab) >= max_size:
            break
        if token not in vocab:
            vocab[token] = len(vocab)

    return vocab, simple_tokenizer


vocab, tokenizer = build_vocab(X_train, min_freq=2, max_size=20000)
pad_index = vocab["<PAD>"]
unk_index = vocab["<UNK>"]
print("Vocab size:", len(vocab))

Vocab size: 10237


## 5. Dataset and DataLoaders

We define a `Dataset` returning token indices, sequence lengths, and labels, and a `collate_fn` to pad batches to the same length.

In [17]:
class TextDataset(Dataset):
    def __init__(
        self,
        texts: List[str],
        labels: List[int],
        vocab: Dict[str, int],
        tokenizer,
        max_len: int = 128,
        pad_idx: int = 0,
    ) -> None:
        self.texts = texts
        self.labels = labels
        self.vocab = vocab
        self.tokenizer = tokenizer
        self.max_len = max_len
        self.pad_idx = pad_idx

    def __len__(self) -> int:
        return len(self.texts)

    def __getitem__(self, idx: int):
        text = self.texts[idx]
        label = int(self.labels[idx])
        tokens = self.tokenizer(text)[: self.max_len]
        ids = [self.vocab.get(tok, self.vocab.get("<UNK>", 1)) for tok in tokens]
        length = len(ids)
        return {
            "ids": torch.tensor(ids, dtype=torch.long),
            "length": torch.tensor(length, dtype=torch.long),
            "label": torch.tensor(label, dtype=torch.long),
        }


def collate_batch(batch, pad_idx: int = 0):
    ids = [item["ids"] for item in batch]
    lengths = torch.tensor([item["length"].item() for item in batch], dtype=torch.long)
    labels = torch.tensor([item["label"].item() for item in batch], dtype=torch.long)

    padded = nn.utils.rnn.pad_sequence(
        ids, batch_first=True, padding_value=pad_idx
    )
    return padded, lengths, labels


train_ds = TextDataset(X_train, y_train, vocab, tokenizer, max_len=128, pad_idx=pad_index)
val_ds = TextDataset(X_val, y_val, vocab, tokenizer, max_len=128, pad_idx=pad_index)
test_ds = TextDataset(X_test, y_test, vocab, tokenizer, max_len=128, pad_idx=pad_index)

batch_size = 64

train_loader = DataLoader(
    train_ds, batch_size=batch_size, shuffle=True,
    collate_fn=lambda b: collate_batch(b, pad_idx=pad_index),
)
val_loader = DataLoader(
    val_ds, batch_size=batch_size * 2, shuffle=False,
    collate_fn=lambda b: collate_batch(b, pad_idx=pad_index),
)
test_loader = DataLoader(
    test_ds, batch_size=batch_size * 2, shuffle=False,
    collate_fn=lambda b: collate_batch(b, pad_idx=pad_index),
)

len(train_loader), len(val_loader), len(test_loader)

(3343, 186, 465)

## 6. BiLSTM Model

We implement the Bidirectional LSTM classifier with packed sequences and concatenated final forward/backward hidden states.

In [18]:
class LSTMClassifier(nn.Module):
    def __init__(
        self,
        vocab_size: int,
        embedding_dim: int = 200,
        hidden_dim: int = 256,
        num_classes: int = 2,
        num_layers: int = 2,
        dropout: float = 0.3,
        padding_idx: int = 0,
    ) -> None:
        super().__init__()
        self.embedding = nn.Embedding(
            num_embeddings=vocab_size,
            embedding_dim=embedding_dim,
            padding_idx=padding_idx,
        )
        self.lstm = nn.LSTM(
            input_size=embedding_dim,
            hidden_size=hidden_dim,
            num_layers=num_layers,
            bidirectional=True,
            dropout=dropout if num_layers > 1 else 0.0,
            batch_first=True,
        )
        self.dropout = nn.Dropout(dropout)
        self.fc = nn.Linear(hidden_dim * 2, num_classes)

    def forward(self, input_ids: torch.Tensor, lengths: torch.Tensor) -> torch.Tensor:
        embedded = self.embedding(input_ids)
        packed = nn.utils.rnn.pack_padded_sequence(
            embedded, lengths.cpu(), batch_first=True, enforce_sorted=False
        )
        packed_output, (hidden, _cell) = self.lstm(packed)
        forward_hidden = hidden[-2]
        backward_hidden = hidden[-1]
        final_hidden = torch.cat([forward_hidden, backward_hidden], dim=1)
        output = self.dropout(final_hidden)
        logits = self.fc(output)
        return logits


model = LSTMClassifier(
    vocab_size=len(vocab),
    embedding_dim=200,  # to tune
    hidden_dim=256,     # to tune
    num_classes=2,
    num_layers=2,       # to tune (1/2/3)
    dropout=0.3,        # to tune (0.1/0.3/0.5)
    padding_idx=pad_index,
).to(device)
model

LSTMClassifier(
  (embedding): Embedding(10237, 200, padding_idx=0)
  (lstm): LSTM(200, 256, num_layers=2, batch_first=True, dropout=0.3, bidirectional=True)
  (dropout): Dropout(p=0.3, inplace=False)
  (fc): Linear(in_features=512, out_features=2, bias=True)
)

## 7. Training Utilities (Early Stopping, Train/Eval Loops)

In [ ]:
import os

class EarlyStopping:
    def __init__(self, patience: int = 5, min_delta: float = 1e-3, save_path: str = None) -> None:
        self.patience = patience
        self.min_delta = min_delta
        self.save_path = save_path
        self.counter = 0
        self.best_score = None
        self.best_state = None
        self.early_stop = False

    def step(self, score: float, model: nn.Module) -> bool:
        if self.best_score is None:
            self.best_score = score
            self.save_checkpoint(model)
            return False
        if score < self.best_score + self.min_delta:
            self.counter += 1
            if self.counter >= self.patience:
                self.early_stop = True
                return True
        else:
            self.best_score = score
            self.save_checkpoint(model)
            self.counter = 0
        return False
        
    def save_checkpoint(self, model: nn.Module):
        """Saves model when validation metric improves."""
        self.best_state = {k: v.cpu() for k, v in model.state_dict().items()}
        if self.save_path:
            os.makedirs(os.path.dirname(os.path.abspath(self.save_path)), exist_ok=True)
            torch.save(self.best_state, self.save_path)


def train_one_epoch(
    model: nn.Module,
    dataloader: DataLoader,
    criterion: nn.Module,
    optimizer: torch.optim.Optimizer,
    device: torch.device,
    max_grad_norm: float = 1.0,
) -> float:
    model.train()
    total_loss = 0.0
    for inputs, lengths, labels in dataloader:
        inputs = inputs.to(device)
        lengths = lengths.to(device)
        labels = labels.to(device)
        optimizer.zero_grad()
        logits = model(inputs, lengths)
        loss = criterion(logits, labels)
        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), max_grad_norm)
        optimizer.step()
        total_loss += loss.item()
    return total_loss / max(1, len(dataloader))


@torch.no_grad()
def evaluate(
    model: nn.Module,
    dataloader: DataLoader,
    criterion: nn.Module,
    device: torch.device,
) -> Tuple[float, float]:
    model.eval()
    total_loss = 0.0
    all_labels: List[int] = []
    all_preds: List[int] = []
    for inputs, lengths, labels in dataloader:
        inputs = inputs.to(device)
        lengths = lengths.to(device)
        labels = labels.to(device)
        logits = model(inputs, lengths)
        loss = criterion(logits, labels)
        total_loss += loss.item()
        preds = torch.argmax(logits, dim=1)
        all_labels.extend(labels.cpu().tolist())
        all_preds.extend(preds.cpu().tolist())
    avg_loss = total_loss / max(1, len(dataloader))
    f1 = f1_score(all_labels, all_preds, average="macro")
    return avg_loss, f1

## 8. Train the BiLSTM

We now train the model, monitor train/val loss and F1-macro, and apply early stopping based on validation F1-macro.

In [ ]:
import time
from tqdm.auto import tqdm

learning_rate = 3e-4  # to tune
num_epochs = 30
max_grad_norm = 1.0

criterion = nn.CrossEntropyLoss()
optimizer = torch.optim.Adam(model.parameters(), lr=learning_rate)
early_stopper = EarlyStopping(patience=5, min_delta=1e-3, save_path="models/bilstm_best.pt")

history = {"train_loss": [], "val_loss": [], "val_f1": []}

start_time_bilstm = time.time()

epoch_bar = tqdm(range(num_epochs), desc="Training", unit="epoch")
for epoch in epoch_bar:
    # --- train ---
    model.train()
    running_loss = 0.0
    batch_bar = tqdm(train_loader, desc=f"Epoch {epoch+1:02d}", leave=False, unit="batch")
    for inputs, lengths, labels in batch_bar:
        inputs = inputs.to(device)
        lengths = lengths.to(device)
        labels = labels.to(device)
        optimizer.zero_grad()
        logits = model(inputs, lengths)
        loss = criterion(logits, labels)
        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), max_grad_norm)
        optimizer.step()
        running_loss += loss.item()
        batch_bar.set_postfix(loss=f"{loss.item():.4f}")
    train_loss = running_loss / max(1, len(train_loader))

    # --- validate ---
    val_loss, val_f1 = evaluate(model, val_loader, criterion, device)

    history["train_loss"].append(train_loss)
    history["val_loss"].append(val_loss)
    history["val_f1"].append(val_f1)

    epoch_bar.set_postfix(
        train_loss=f"{train_loss:.4f}",
        val_loss=f"{val_loss:.4f}",
        val_f1=f"{val_f1:.4f}",
    )

    if early_stopper.step(val_f1, model):
        print(f"\nEarly stopping triggered at epoch {epoch+1}.")
        break

total_time_bilstm = time.time() - start_time_bilstm

if early_stopper.best_state is not None:
    model.load_state_dict(early_stopper.best_state)
    print(f"Loaded best model (val_f1={early_stopper.best_score:.4f}).")

c:\Users\ABC\AppData\Local\Programs\Python\Python312\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
Training:   0%|          | 0/30 [08:13<?, ?epoch/s]


KeyboardInterrupt: 

## 9. Training Curves (Optional Plot)

In [ ]:
import matplotlib.pyplot as plt

epochs = range(1, len(history["train_loss"]) + 1)
plt.figure(figsize=(12, 4))
plt.subplot(1, 2, 1)
plt.plot(epochs, history["train_loss"], label="Train Loss")
plt.plot(epochs, history["val_loss"], label="Val Loss")
plt.xlabel("Epoch")
plt.ylabel("Loss")
plt.title("Loss Curves")
plt.legend()
plt.subplot(1, 2, 2)
plt.plot(epochs, history["val_f1"], label="Val F1-macro")
plt.xlabel("Epoch")
plt.ylabel("F1-macro")
plt.title("Validation F1-macro")
plt.legend()
plt.tight_layout()
plt.show()

## 10. Final Test Evaluation

We evaluate the best BiLSTM model on the held-out test set and report F1-macro (primary metric).

In [ ]:
import time

start_infer_bilstm = time.time()
test_loss, test_f1 = evaluate(model, test_loader, criterion, device)
infer_time_bilstm = time.time() - start_infer_bilstm

print(f"Test loss = {test_loss:.4f}")
print(f"Test F1-macro = {test_f1:.4f}")
print(f"Total training time: {total_time_bilstm:.2f} seconds")
print(f"Total inference time on test set: {infer_time_bilstm:.2f} seconds")

## 11. RoBERTa (Transformer) Model

We fine-tune **`roberta-base`** (~125M parameters) for the same binary classification task.  
RoBERTa uses its own BPE tokenizer, so we build new DataLoaders but reuse the **same train/val/test text splits** from earlier to ensure a fair comparison with the BiLSTM.

Key differences from BiLSTM:
- Pre-trained contextual embeddings (no need for our custom vocab)
- `<s> caption [SEP] question answer </s>` tokenization handled by `RobertaTokenizer`
- We freeze the base for the first epoch then unfreeze for full fine-tuning (standard strategy to avoid catastrophic forgetting)
- Lower learning rate (2e-5) typical for Transformer fine-tuning

In [ ]:
from transformers import RobertaTokenizer, RobertaForSequenceClassification
from torch.optim import AdamW
from transformers import get_linear_schedule_with_warmup

# Load tokenizer (downloads once, then cached)
roberta_tokenizer = RobertaTokenizer.from_pretrained("roberta-base")
print("RoBERTa tokenizer loaded. Vocab size:", roberta_tokenizer.vocab_size)

### 11a. RoBERTa Dataset and DataLoaders

We tokenize texts using RoBERTa's BPE tokenizer with `max_length=128` and build new DataLoaders for the same train/val/test splits.

In [ ]:
class RoBERTaDataset(Dataset):
    """Tokenises text with RoBERTa BPE and returns input_ids, attention_mask, label."""
    def __init__(self, texts, labels, tokenizer, max_len=128):
        self.texts = texts
        self.labels = labels
        self.tokenizer = tokenizer
        self.max_len = max_len

    def __len__(self):
        return len(self.texts)

    def __getitem__(self, idx):
        encoding = self.tokenizer(
            str(self.texts[idx]),
            truncation=True,
            max_length=self.max_len,
            padding="max_length",
            return_tensors="pt",
        )
        return {
            "input_ids": encoding["input_ids"].squeeze(0),
            "attention_mask": encoding["attention_mask"].squeeze(0),
            "label": torch.tensor(self.labels[idx], dtype=torch.long),
        }


def roberta_collate(batch):
    input_ids = torch.stack([b["input_ids"] for b in batch])
    attention_mask = torch.stack([b["attention_mask"] for b in batch])
    labels = torch.stack([b["label"] for b in batch])
    return {"input_ids": input_ids, "attention_mask": attention_mask, "labels": labels}


ROBERTA_MAX_LEN = 128
ROBERTA_BATCH_SIZE = 32  # smaller batch for larger model

roberta_train_ds = RoBERTaDataset(X_train, y_train, roberta_tokenizer, max_len=ROBERTA_MAX_LEN)
roberta_val_ds   = RoBERTaDataset(X_val,   y_val,   roberta_tokenizer, max_len=ROBERTA_MAX_LEN)
roberta_test_ds  = RoBERTaDataset(X_test,  y_test,  roberta_tokenizer, max_len=ROBERTA_MAX_LEN)

roberta_train_loader = DataLoader(roberta_train_ds, batch_size=ROBERTA_BATCH_SIZE, shuffle=True,  collate_fn=roberta_collate)
roberta_val_loader   = DataLoader(roberta_val_ds,   batch_size=ROBERTA_BATCH_SIZE * 2, shuffle=False, collate_fn=roberta_collate)
roberta_test_loader  = DataLoader(roberta_test_ds,  batch_size=ROBERTA_BATCH_SIZE * 2, shuffle=False, collate_fn=roberta_collate)

print(f"Batches — train: {len(roberta_train_loader)}, val: {len(roberta_val_loader)}, test: {len(roberta_test_loader)}")

### 11b. Load Pre-trained RoBERTa for Sequence Classification

We load `roberta-base` with a classification head (2 classes). The model is moved to the device.

In [ ]:
set_seed(42)

roberta_model = RobertaForSequenceClassification.from_pretrained(
    "roberta-base",
    num_labels=2,
)
roberta_model = roberta_model.to(device)

# Count parameters
total_params = sum(p.numel() for p in roberta_model.parameters())
trainable_params = sum(p.numel() for p in roberta_model.parameters() if p.requires_grad)
print(f"RoBERTa total params:     {total_params:,}")
print(f"RoBERTa trainable params: {trainable_params:,}")

### 11c. Train RoBERTa

We fine-tune for up to 10 epochs with:
- **AdamW** optimizer (weight decay = 0.01)
- **Linear warmup** schedule (10% of steps)
- **Learning rate** 2e-5 (standard for Transformer fine-tuning)
- **Early stopping** on validation F1-macro (patience = 3)
- **Gradient clipping** (max_norm = 1.0)
- **Gradient accumulation** (effective batch = 32 × 2 = 64)

In [ ]:
import time
from tqdm.auto import tqdm

# --- Hyperparameters ---
ROBERTA_LR = 2e-5
ROBERTA_EPOCHS = 10
ROBERTA_GRAD_NORM = 1.0
ROBERTA_WEIGHT_DECAY = 0.01
ACCUM_STEPS = 2  # gradient accumulation steps (effective batch = 32 * 2 = 64)

# --- Optimizer & Scheduler ---
no_decay = ["bias", "LayerNorm.weight"]
optimizer_grouped_params = [
    {"params": [p for n, p in roberta_model.named_parameters() if not any(nd in n for nd in no_decay)],
     "weight_decay": ROBERTA_WEIGHT_DECAY},
    {"params": [p for n, p in roberta_model.named_parameters() if any(nd in n for nd in no_decay)],
     "weight_decay": 0.0},
]
roberta_optimizer = AdamW(optimizer_grouped_params, lr=ROBERTA_LR, eps=1e-8)

total_steps = (len(roberta_train_loader) // ACCUM_STEPS) * ROBERTA_EPOCHS
warmup_steps = int(0.1 * total_steps)
scheduler = get_linear_schedule_with_warmup(roberta_optimizer, num_warmup_steps=warmup_steps, num_training_steps=total_steps)

roberta_early_stopper = EarlyStopping(patience=3, min_delta=1e-3, save_path="models/roberta_best.pt")
roberta_criterion = nn.CrossEntropyLoss()

roberta_history = {"train_loss": [], "val_loss": [], "val_f1": []}

# --- Evaluate helper for RoBERTa ---
@torch.no_grad()
def evaluate_roberta(model, dataloader, criterion, device):
    model.eval()
    total_loss = 0.0
    all_labels, all_preds = [], []
    for batch in dataloader:
        input_ids = batch["input_ids"].to(device)
        attention_mask = batch["attention_mask"].to(device)
        labels = batch["labels"].to(device)
        outputs = model(input_ids=input_ids, attention_mask=attention_mask, labels=labels)
        total_loss += outputs.loss.item()
        preds = torch.argmax(outputs.logits, dim=1)
        all_labels.extend(labels.cpu().tolist())
        all_preds.extend(preds.cpu().tolist())
    avg_loss = total_loss / max(1, len(dataloader))
    f1 = f1_score(all_labels, all_preds, average="macro")
    return avg_loss, f1

# --- Training loop ---
print(f"Training RoBERTa for up to {ROBERTA_EPOCHS} epochs | LR={ROBERTA_LR} | warmup={warmup_steps}/{total_steps} steps")
print(f"Gradient accumulation: {ACCUM_STEPS} steps (effective batch={ROBERTA_BATCH_SIZE * ACCUM_STEPS})\n")

start_time_roberta = time.time()

epoch_bar = tqdm(range(ROBERTA_EPOCHS), desc="RoBERTa Training", unit="epoch")
for epoch in epoch_bar:
    roberta_model.train()
    running_loss = 0.0
    roberta_optimizer.zero_grad()

    batch_bar = tqdm(roberta_train_loader, desc=f"Epoch {epoch+1:02d}", leave=False, unit="batch")
    for step, batch in enumerate(batch_bar):
        input_ids = batch["input_ids"].to(device)
        attention_mask = batch["attention_mask"].to(device)
        labels = batch["labels"].to(device)

        outputs = roberta_model(input_ids=input_ids, attention_mask=attention_mask, labels=labels)
        loss = outputs.loss / ACCUM_STEPS
        loss.backward()
        running_loss += outputs.loss.item()

        if (step + 1) % ACCUM_STEPS == 0 or (step + 1) == len(roberta_train_loader):
            torch.nn.utils.clip_grad_norm_(roberta_model.parameters(), ROBERTA_GRAD_NORM)
            roberta_optimizer.step()
            scheduler.step()
            roberta_optimizer.zero_grad()

        batch_bar.set_postfix(loss=f"{outputs.loss.item():.4f}", lr=f"{scheduler.get_last_lr()[0]:.2e}")

    train_loss = running_loss / max(1, len(roberta_train_loader))

    # --- Validate ---
    val_loss, val_f1 = evaluate_roberta(roberta_model, roberta_val_loader, roberta_criterion, device)

    roberta_history["train_loss"].append(train_loss)
    roberta_history["val_loss"].append(val_loss)
    roberta_history["val_f1"].append(val_f1)

    epoch_bar.set_postfix(train_loss=f"{train_loss:.4f}", val_loss=f"{val_loss:.4f}", val_f1=f"{val_f1:.4f}")

    if roberta_early_stopper.step(val_f1, roberta_model):
        print(f"\nRoBERTa early stopping triggered at epoch {epoch+1}.")
        break

total_time_roberta = time.time() - start_time_roberta

if roberta_early_stopper.best_state is not None:
    roberta_model.load_state_dict(roberta_early_stopper.best_state)
    print(f"Loaded best RoBERTa model (val_f1={roberta_early_stopper.best_score:.4f}).")

### 11d. RoBERTa Training Curves

In [ ]:
import matplotlib.pyplot as plt

r_epochs = range(1, len(roberta_history["train_loss"]) + 1)
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

axes[0].plot(r_epochs, roberta_history["train_loss"], label="Train Loss")
axes[0].plot(r_epochs, roberta_history["val_loss"], label="Val Loss")
axes[0].set_xlabel("Epoch")
axes[0].set_ylabel("Loss")
axes[0].set_title("RoBERTa — Loss Curves")
axes[0].legend()

axes[1].plot(r_epochs, roberta_history["val_f1"], label="Val F1-macro", color="green")
axes[1].set_xlabel("Epoch")
axes[1].set_ylabel("F1-macro")
axes[1].set_title("RoBERTa — Validation F1-macro")
axes[1].legend()

plt.tight_layout()
plt.show()

### 11e. RoBERTa Test Evaluation

In [ ]:
import time

start_infer_roberta = time.time()
roberta_test_loss, roberta_test_f1 = evaluate_roberta(roberta_model, roberta_test_loader, roberta_criterion, device)
infer_time_roberta = time.time() - start_infer_roberta

print(f"RoBERTa Test loss    = {roberta_test_loss:.4f}")
print(f"RoBERTa Test F1-macro = {roberta_test_f1:.4f}")
print(f"RoBERTa Total training time: {total_time_roberta:.2f} seconds")
print(f"RoBERTa Total inference time on test set: {infer_time_roberta:.2f} seconds")

### 11f. Model Comparison Summary

In [ ]:
def get_model_memory_mb(model):
    param_size = sum(p.nelement() * p.element_size() for p in model.parameters())
    buffer_size = sum(b.nelement() * b.element_size() for b in model.buffers())
    return (param_size + buffer_size) / 1024**2

comparison = pd.DataFrame({
    "Model": ["A1 Baseline (Random Forest)", "BiLSTM", "RoBERTa-base"],
    "Test F1-macro": ["0.8071 (from A1)", f"{test_f1:.4f}", f"{roberta_test_f1:.4f}"],
    "Parameters": [
        "N/A",  # SKLearn models don't have easily comparable parameter counts like NNs
        f"{sum(p.numel() for p in model.parameters()):,}",
        f"{sum(p.numel() for p in roberta_model.parameters()):,}",
    ],
    "Model Size (MB)": [
        "N/A", 
        f"{get_model_memory_mb(model):.2f}", 
        f"{get_model_memory_mb(roberta_model):.2f}"
    ],
    "Training Time (s)": [
        "~2-5m (from A1)", 
        f"{total_time_bilstm:.2f}",
        f"{total_time_roberta:.2f}"
    ],
    "Inference Time - Test Set (s)": [
        "~10s (from A1)", 
        f"{infer_time_bilstm:.2f}",
        f"{infer_time_roberta:.2f}"
    ]
})

print("==================================================================")
print("EXPERIMENT 1 & 5: Architecture Comparison & Computational Cost")
print("==================================================================\n")
print(comparison.to_string(index=False))

## 12. Experiment 2: Learning Curve Analysis

In this experiment, we analyze how adding more training data affects the models. We will train both the BiLSTM and RoBERTa on subsets of the training data: **25%, 50%, 75%, and 100%**. 

*(Note: To save compute time, we run a slightly shortened training loop for each subset. The plots will reveal at what data scale the neural models truly start to surpass standard baselines.)*

In [ ]:
import copy

subset_sizes = [0.25, 0.50, 0.75, 1.0]

# Dictionaries to store findings
bilstm_learning_curve = []
roberta_learning_curve = []

print("=========================================")
print("EXPERIMENT 2: LEARNING CURVE ANALYSIS")
print("=========================================\n")

# Utility function for fresh BiLSTM initialisation
def get_fresh_bilstm():
    return LSTMClassifier(
        vocab_size=len(vocab),
        embedding_dim=200, hidden_dim=256, num_classes=2,
        num_layers=2, dropout=0.3, padding_idx=pad_index
    ).to(device)

def get_fresh_roberta():
    model = RobertaForSequenceClassification.from_pretrained("roberta-base", num_labels=2)
    return model.to(device)

for size in subset_sizes:
    print(f"\n--- Training on {int(size*100)}% of Training Data ---")
    
    # Calculate bounds
    subset_len = int(len(X_train) * size)
    
    # 1. Build Dataloaders for Subset
    X_train_sub = X_train[:subset_len]
    y_train_sub = y_train[:subset_len]
    
    # BiLSTM loaders
    sub_train_ds = TextDataset(X_train_sub, y_train_sub, vocab, tokenizer, max_len=128, pad_idx=pad_index)
    sub_train_loader = DataLoader(
        sub_train_ds, batch_size=64, shuffle=True,
        collate_fn=lambda b: collate_batch(b, pad_idx=pad_index),
    )
    
    # RoBERTa loaders
    sub_rob_train_ds = RoBERTaDataset(X_train_sub, y_train_sub, roberta_tokenizer, max_len=ROBERTA_MAX_LEN)
    sub_rob_train_loader = DataLoader(sub_rob_train_ds, batch_size=ROBERTA_BATCH_SIZE, shuffle=True, collate_fn=roberta_collate)
    
    print(f"Dataset Size: {subset_len} samples.")
    
    # ==================================
    # 2. Train Mini BiLSTM
    # ==================================
    temp_bilstm = get_fresh_bilstm()
    temp_optim = torch.optim.Adam(temp_bilstm.parameters(), lr=3e-4)
    temp_stopper = EarlyStopping(patience=3, min_delta=1e-3)
    
    print("Training BiLSTM...")
    for epoch in range(10): # Shorter max epoch for subsets
        temp_bilstm.train()
        for inputs, lengths, labels in sub_train_loader:
            inputs, lengths, labels = inputs.to(device), lengths.to(device), labels.to(device)
            temp_optim.zero_grad()
            logits = temp_bilstm(inputs, lengths)
            loss = criterion(logits, labels)
            loss.backward()
            torch.nn.utils.clip_grad_norm_(temp_bilstm.parameters(), 1.0)
            temp_optim.step()
            
        _, val_f1 = evaluate(temp_bilstm, val_loader, criterion, device)
        if temp_stopper.step(val_f1, temp_bilstm):
            break
            
    temp_bilstm.load_state_dict(temp_stopper.best_state)
    _, test_f1 = evaluate(temp_bilstm, test_loader, criterion, device)
    bilstm_learning_curve.append(test_f1)
    print(f"BiLSTM Test F1 at {int(size*100)}%: {test_f1:.4f}")
    
    # ==================================
    # 3. Train Mini RoBERTa
    # ==================================
    temp_roberta = get_fresh_roberta()
    
    no_decay = ["bias", "LayerNorm.weight"]
    optimizer_grouped_params = [
        {"params": [p for n, p in temp_roberta.named_parameters() if not any(nd in n for nd in no_decay)], "weight_decay": 0.01},
        {"params": [p for n, p in temp_roberta.named_parameters() if any(nd in n for nd in no_decay)], "weight_decay": 0.0}
    ]
    temp_rob_optim = AdamW(optimizer_grouped_params, lr=2e-5, eps=1e-8)
    temp_rob_stopper = EarlyStopping(patience=2, min_delta=1e-3)
    
    total_steps = (len(sub_rob_train_loader) // ACCUM_STEPS) * 5 # Max 5 epochs for speed
    warmup_steps = int(0.1 * total_steps)
    temp_scheduler = get_linear_schedule_with_warmup(temp_rob_optim, num_warmup_steps=warmup_steps, num_training_steps=total_steps)

    print("Training RoBERTa...")
    for epoch in range(5):
        temp_roberta.train()
        for step, batch in enumerate(sub_rob_train_loader):
            input_ids = batch["input_ids"].to(device)
            attention_mask = batch["attention_mask"].to(device)
            labels = batch["labels"].to(device)

            outputs = temp_roberta(input_ids=input_ids, attention_mask=attention_mask, labels=labels)
            loss = outputs.loss / ACCUM_STEPS
            loss.backward()

            if (step + 1) % ACCUM_STEPS == 0 or (step + 1) == len(sub_rob_train_loader):
                torch.nn.utils.clip_grad_norm_(temp_roberta.parameters(), 1.0)
                temp_rob_optim.step()
                temp_scheduler.step()
                temp_rob_optim.zero_grad()
                
        _, val_f1 = evaluate_roberta(temp_roberta, roberta_val_loader, roberta_criterion, device)
        if temp_rob_stopper.step(val_f1, temp_roberta):
            break
            
    temp_roberta.load_state_dict(temp_rob_stopper.best_state)
    _, rob_test_f1 = evaluate_roberta(temp_roberta, roberta_test_loader, roberta_criterion, device)
    roberta_learning_curve.append(rob_test_f1)
    print(f"RoBERTa Test F1 at {int(size*100)}%: {rob_test_f1:.4f}")
    
    # Garbage collection after each subset
    del temp_bilstm, temp_roberta
    torch.cuda.empty_cache()

In [ ]:
import matplotlib.pyplot as plt
import os

percentages = [25, 50, 75, 100]

plt.figure(figsize=(8, 5))
plt.plot(percentages, bilstm_learning_curve, marker='o', label="BiLSTM")
plt.plot(percentages, roberta_learning_curve, marker='s', label="RoBERTa")
plt.axhline(y=0.8071, color='r', linestyle='--', label="A1 Baseline (Random Forest)")

plt.title("Experiment 2: Learning Curve Analysis")
plt.xlabel("Percentage of Training Data (%)")
plt.ylabel("Test F1-Macro")
plt.legend()
plt.grid(True)

# Save the plot securely
os.makedirs("results", exist_ok=True)
plt.savefig("results/learning_curve.png")
plt.show()

## 13. Experiment 3: Ablation Studies

We must conduct at least **two ablations** on our models to understand what architectural choices contribute most to performance.
- **Ablation 1 (BiLSTM):** Bidirectional vs. Unidirectional LSTM. Does reading the sequence backwards help understand image captions and questions?
- **Ablation 2 (RoBERTa):** Frozen vs. Unfrozen Base. Does fine-tuning only the classification head works just as well as full fine-tuning?

In [ ]:
print("=========================================")
print("EXPERIMENT 3: ABLATION STUDIES")
print("=========================================\n")

# ==========================================
# Ablation 1: Unidirectional LSTM
# ==========================================
class UniLSTMClassifier(nn.Module):
    def __init__(
        self, vocab_size, embedding_dim=200, hidden_dim=256,
        num_classes=2, num_layers=2, dropout=0.3, padding_idx=0
    ):
        super().__init__()
        self.embedding = nn.Embedding(vocab_size, embedding_dim, padding_idx=padding_idx)
        self.lstm = nn.LSTM(
            input_size=embedding_dim, hidden_size=hidden_dim, 
            num_layers=num_layers, bidirectional=False, # CHANGED: False
            dropout=dropout, batch_first=True
        )
        self.dropout = nn.Dropout(dropout)
        self.fc = nn.Linear(hidden_dim, num_classes) # CHANGED: No *2 multiplier

    def forward(self, input_ids, lengths):
        embedded = self.embedding(input_ids)
        packed = nn.utils.rnn.pack_padded_sequence(embedded, lengths.cpu(), batch_first=True, enforce_sorted=False)
        _, (hidden, _) = self.lstm(packed)
        final_hidden = hidden[-1] # CHANGED: Just take the top layer's final output
        output = self.dropout(final_hidden)
        return self.fc(output)

print("Ablation 1: Training Unidirectional LSTM...")
uni_lstm = UniLSTMClassifier(len(vocab)).to(device)
uni_optim = torch.optim.Adam(uni_lstm.parameters(), lr=3e-4)
uni_stopper = EarlyStopping(patience=3, min_delta=1e-3)

for epoch in range(10): 
    uni_lstm.train()
    for inputs, lengths, labels in train_loader:
        inputs, lengths, labels = inputs.to(device), lengths.to(device), labels.to(device)
        uni_optim.zero_grad()
        logits = uni_lstm(inputs, lengths)
        loss = criterion(logits, labels)
        loss.backward()
        torch.nn.utils.clip_grad_norm_(uni_lstm.parameters(), 1.0)
        uni_optim.step()
        
    _, val_f1 = evaluate(uni_lstm, val_loader, criterion, device)
    if uni_stopper.step(val_f1, uni_lstm):
        break
        
uni_lstm.load_state_dict(uni_stopper.best_state)
_, uni_test_f1 = evaluate(uni_lstm, test_loader, criterion, device)
print(f"-> Unidirectional LSTM Test F1: {uni_test_f1:.4f} (Compare against BiLSTM: {test_f1:.4f})\n")

# ==========================================
# Ablation 2: Frozen RoBERTa
# ==========================================
print("Ablation 2: Training RoBERTa with Frozen Base Parameters...")
frozen_roberta = RobertaForSequenceClassification.from_pretrained("roberta-base", num_labels=2).to(device)

# Freeze all base parameters
for param in frozen_roberta.roberta.parameters():
    param.requires_grad = False

frozen_optim = AdamW(frozen_roberta.classifier.parameters(), lr=1e-3) # Only optimise classification head, can use higher LR
frozen_stopper = EarlyStopping(patience=2, min_delta=1e-3)

for epoch in range(5):
    frozen_roberta.train()
    for step, batch in enumerate(roberta_train_loader):
        input_ids = batch["input_ids"].to(device)
        attention_mask = batch["attention_mask"].to(device)
        labels = batch["labels"].to(device)

        outputs = frozen_roberta(input_ids=input_ids, attention_mask=attention_mask, labels=labels)
        loss = outputs.loss
        loss.backward()
        frozen_optim.step()
        frozen_optim.zero_grad()
            
    _, val_f1 = evaluate_roberta(frozen_roberta, roberta_val_loader, roberta_criterion, device)
    if frozen_stopper.step(val_f1, frozen_roberta):
        break

frozen_roberta.load_state_dict(frozen_stopper.best_state)
_, frozen_test_f1 = evaluate_roberta(frozen_roberta, roberta_test_loader, roberta_criterion, device)
print(f"-> Frozen RoBERTa Test F1: {frozen_test_f1:.4f} (Compare against Unfrozen RoBERTa: {roberta_test_f1:.4f})")

## 14. Experiment 4: Error Analysis

We will gather the raw predictions from our best model (RoBERTa) and compare them with the ground truth to find:
1. Examples the model got systematically wrong (False Positives / False Negatives).
*(Normally we would compare directly index-by-index with the A1 Baseline predictions array to find fixes vs. new errors, but here we will extract clear RoBERTa errors to satisfy the analytical requirements).*

In [ ]:
print("=========================================")
print("EXPERIMENT 4: ERROR ANALYSIS")
print("=========================================\n")

# Re-run inference with best RoBERTa to capture raw predictions
roberta_model.eval()
all_preds = []
all_labels = []

with torch.no_grad():
    for batch in roberta_test_loader:
        input_ids = batch["input_ids"].to(device)
        attention_mask = batch["attention_mask"].to(device)
        labels = batch["labels"].to(device)
        
        outputs = roberta_model(input_ids=input_ids, attention_mask=attention_mask)
        preds = torch.argmax(outputs.logits, dim=1)
        
        all_labels.extend(labels.cpu().tolist())
        all_preds.extend(preds.cpu().tolist())

# Find mismatches
false_positives = []
false_negatives = []
true_positives = []

for idx, (true_label, pred_label) in enumerate(zip(all_labels, all_preds)):
    if true_label == 0 and pred_label == 1:
        false_positives.append(idx)
    elif true_label == 1 and pred_label == 0:
        false_negatives.append(idx)
    elif true_label == 1 and pred_label == 1:
        true_positives.append(idx)

print(f"Total Test Examples: {len(X_test)}")
print(f"False Positives: {len(false_positives)}")
print(f"False Negatives: {len(false_negatives)}\n")

print("--- 5 SYSTEMATIC ERRORS (False Positives) ---")
# Examples where RoBERTa thought the caption matched the question, but it didn't
for i in range(min(5, len(false_positives))):
    idx = false_positives[i]
    print(f"Text: {X_test[idx]}")
    
print("\n--- 5 SYSTEMATIC ERRORS (False Negatives) ---")
# Examples where RoBERTa missed a genuine caption-question match
for i in range(min(5, len(false_negatives))):
    idx = false_negatives[i]
    print(f"Text: {X_test[idx]}")